#### QUERY AND VALIDATE `GIZMO.BRONZE.ADDRESSES_VW`

In [0]:
address_df = spark.table('''GIZMO.BRONZE.ADDRESSES_VW''')
display(address_df)

In [0]:
from pyspark.sql import functions as F

# 2. Perform the Pivot
pivot_address_df = address_df.groupBy("customer_id").pivot("address_type", ["shipping", "billing"]).agg(
    F.max("address_line_1").alias("address_line_1"),
    F.max("city").alias("city"),
    F.max("state").alias("state"),
    F.max("postcode").alias("postcode")
)

display(pivot_address_df)

In [0]:
# 3. Rename columns to match your SQL output (Optional)
# PySpark defaults to 'shipping_address_line_1', but sometimes 
# needs explicit renaming depending on the Spark version.

pivot_address_final_df = pivot_address_df.select(
    "customer_id",
    "shipping_address_line_1",
    "shipping_city",
    "shipping_state",
    "shipping_postcode",
    "billing_address_line_1",
    "billing_city",
    "billing_state",
    "billing_postcode"
)

display(pivot_address_final_df)

#### WRITE TRANSFORMED DATA TO SILVER SCHEMA
1. CATALOG NAME: GIZMO
2. SCHEMA NAME: SILVER
3. TABLE NAME: ADDRESSES_DELTA

In [0]:
pivot_address_final_df.write.mode('overwrite').format('delta').saveAsTable('gizmo.silver.addresses_delta')

#### VALIDATE `GIZMO.SILVER.ADDRESSES_DELTA`

In [0]:
%sql
SELECT * FROM GIZMO.SILVER.ADDRESSES_DELTA LIMIT 1;

In [0]:
%python
dbutils.notebook.exit("ADDRESSES LOADED INTO GIZMO.SILVER.ADDRESSES_DELTA")
